In [1]:
## Codice paper 
from array import array
from scipy.sparse import *
import gc
import networkx as nx


class SignedGraph:

    def __init__(self, dataset_path):
        # nodes
        self.number_of_nodes = 0
        self.nodes_iterator = range(0)

        # adjacency list
        self.adjacency_list = []

        # adjacency matrix and laplacian
        self.a = None
        self.l = None

        # load the dataset
        self.load_dataset(dataset_path)

        # set the dataset path
        self.dataset_path = dataset_path

        # call the garbage collector
        gc.collect()

    def load_dataset(self, dataset_path):
        # open the file
        try:
            dataset_file = open('../datasets/' + dataset_path + '.txt')
        except IOError:
            dataset_file = open('../../datasets/' + dataset_path + '.txt')

        # get the number of nodes from the first line
        self.number_of_nodes = int(dataset_file.readline().replace('# ', ''))
        self.nodes_iterator = range(self.number_of_nodes)

        # create the empty adjacency list
        self.adjacency_list = [[array('i'), array('i')] for _ in self.nodes_iterator]

        # fill the adjacency matrix (0: positive neighbors, 1: negative neighbors)
        for line in dataset_file:
            split_line = line.split('\t')
            from_node = int(split_line[0])
            to_node = int(split_line[1])
            sign = int(split_line[2])

            # add the undirected edge
            self.add_edge(from_node, to_node, sign)

    # add the edge to the adjacency list if it is not a self loop
    def add_edge(self, from_node, to_node, sign):
        if sign == 1:
            sign = 0
        else:
            sign = 1

        if from_node != to_node:
            self.adjacency_list[from_node][sign].append(to_node)
            self.adjacency_list[to_node][sign].append(from_node)

    def get_adjacency_matrix(self):
        if self.a is None:
            self.a = lil_matrix((self.number_of_nodes, self.number_of_nodes), dtype='d')

            for node, neighbors in enumerate(self.adjacency_list):
                for neighbor in neighbors[0]:
                    self.a[node, neighbor] = 1
                for neighbor in neighbors[1]:
                    self.a[node, neighbor] = -1

            self.a = self.a.tocsr()

        return self.a

    def get_signed_laplacian(self):
        if self.l is None:
            self.l = lil_matrix((self.number_of_nodes, self.number_of_nodes), dtype='d')

            # add the inverted signs and the degree on the diagonal
            for node, neighbors in enumerate(self.adjacency_list):
                for neighbor in neighbors[0]:
                    self.l[node, neighbor] = -1
                for neighbor in neighbors[1]:
                    self.l[node, neighbor] = 1
                self.l[node, node] = len(neighbors[0]) + len(neighbors[1])

            self.l = self.l.tocsr()

        return self.l

    def get_signed_laplacian_subgraph(self, nodes):
        # order the nodes
        nodes = list(nodes)
        nodes.sort()

        # get the ordering in a map
        ordering = dict(zip(nodes, range(len(nodes))))

        rows, columns, data = [], [], []
        for node in nodes:
            order = ordering[node]
            degree = 0
            for neighbor in self.adjacency_list[node][0]:
                if neighbor in ordering:
                    rows.append(order)
                    columns.append(ordering[neighbor])
                    data.append(-1)
                    degree += 1
            for neighbor in self.adjacency_list[node][1]:
                if neighbor in ordering:
                    rows.append(order)
                    columns.append(ordering[neighbor])
                    data.append(1)
                    degree += 1
            rows.append(order)
            columns.append(order)
            data.append(degree)

        return coo_matrix((data, (rows, columns)), shape=(len(nodes), len(nodes)), dtype='d').tocsr()

    ### Nuova funzione to_networkx per convertire in grafo NetworkX, in modo da
    ### poter utilizzare funzioni e metodi già esistenti di utilità
    def to_networkx(self):
        """
        Converte l'oggetto SignedGraph in un grafo NetworkX non orientato.
        - Nodi: 0..number_of_nodes-1
        - Archi: con attributo 'sign' = +1 (positivo) o -1 (negativo)
        """
        G = nx.Graph()
        G.add_nodes_from(range(self.number_of_nodes))

        # scorro i nodi
        for u, (pos_neighbors, neg_neighbors) in enumerate(self.adjacency_list):
            # positivi
            for v in pos_neighbors:
                if u < v:  # evita duplicati
                    G.add_edge(u, v, sign=+1)
            # negativi
            for v in neg_neighbors:
                if u < v:
                    G.add_edge(u, v, sign=-1)
                    

        return G
#### Copia della repository

from scipy.sparse.linalg import eigsh
import numpy as np


def evaluate_objective_function(signed_graph, x):
    # special case with no nodes in the solution
    if x.dot(x) == 0:
        return np.nan

    # obtain the adjacency matrix
    a = signed_graph.get_adjacency_matrix()

    # compute the objective function
    a_dot_x = a.dot(x)
    return x.dot(a_dot_x) / x.dot(x)


def build_solution(x):
    # return the nodes having the corresponding index of x different from 0
    return {node for node, element in enumerate(x) if element != 0}


def build_x(signed_graph, nodes, eigenvector=None):
    # get the maximum eigenvector of the adjacency matrix
    if eigenvector is None:
        a = signed_graph.get_adjacency_matrix()
        eigenvector = np.squeeze(eigsh(a, k=1, which='LA')[1])

    # build x from the signs of the minimum eigenvector
    return np.array([np.sign(element) if node in nodes else 0 for node, element in enumerate(eigenvector)])

from algorithms.subroutines.commons import *
from utilities.time_measure import ExecutionTime
from utilities.print_console import print_end_algorithm


def eigensign(signed_graph):
    # start of the algorithm
    execution_time = ExecutionTime()

    # initialize the solution as empty
    solution_x = None
    solution_objective_function = np.finfo(float).min
    solution_threshold = None

    # obtain the adjacency matrix
    a = signed_graph.get_adjacency_matrix()

    # get the eigenvector corresponding to the maximum eigenvalue
    maximum_eigenvector = np.squeeze(eigsh(a, k=1, which='LA')[1])

    # get the thresholds from the eigenvector
    thresholds = {int(np.abs(element) * 1000) / 1000.0 for element in maximum_eigenvector}

    # compute x for all the values of the threshold
    for threshold in thresholds:
        x = np.array([np.sign(element) if np.abs(element) >= threshold else 0 for element in maximum_eigenvector])

        # update the solution if needed
        objective_function = evaluate_objective_function(signed_graph, x)
        if objective_function > solution_objective_function:
            solution_x = x
            solution_objective_function = objective_function
            solution_threshold = threshold

    # build the solution
    solution = build_solution(solution_x)

    # end of the algorithm
    execution_time.end_algorithm()

    # print algorithm's results
    print_end_algorithm(execution_time.execution_time_seconds, solution_x, signed_graph, threshold=solution_threshold)

    # return the solution
    return solution, solution_x


from algorithms.subroutines.commons import *
from utilities.time_measure import ExecutionTime
from utilities.print_console import print_end_algorithm


def random_eigensign(signed_graph, beta, maximum_eigenvector=None, execution_time_seconds=None):
    # start of the algorithm
    execution_time = ExecutionTime()

    if maximum_eigenvector is None:
        # obtain the adjacency matrix
        a = signed_graph.get_adjacency_matrix()

        # get the eigenvector corresponding to the maximum eigenvalue
        maximum_eigenvector = np.squeeze(eigsh(a, k=1, which='LA')[1])

        # consolidate beta
        if beta == 'l1':
            beta = np.linalg.norm(maximum_eigenvector, ord=1)
        elif beta == 'sqrt':
            beta = np.sqrt(signed_graph.number_of_nodes)
        else:
            beta = float(beta)

        # multiply the maximum eigenvector by beta
        maximum_eigenvector *= beta

    # compute x
    x = np.array([0 for _ in signed_graph.nodes_iterator])
    for node, element in enumerate(maximum_eigenvector):
        # check the probability for a certain number of times
        if np.random.choice((True, False), p=(min(np.abs(element), 1), max(1 - np.abs(element), 0))):
            x[node] = np.sign(element)

    # build the solution
    solution = build_solution(x)

    # end of the algorithm
    execution_time.end_algorithm()

    # print algorithm's results
    if execution_time_seconds is None:
        execution_time_seconds = execution_time.execution_time_seconds
    print_end_algorithm(execution_time_seconds, x, signed_graph, beta=beta)

    # return the solution
    return solution, x, maximum_eigenvector, execution_time_seconds, beta

### Prove grafo segnato dai dataset
# DATASET_PATH = "twitterreferendum"

# read the input graph
# signed_graph = SignedGraph(DATASET_PATH)
# G = signed_graph.to_networkx()
# print(G.number_of_nodes(), G.number_of_edges())
# Conta archi positivi e negativi
# positive_edges = sum(1 for _, _, data in G.edges(data=True) if data['sign'] == 1)
# negative_edges = sum(1 for _, _, data in G.edges(data=True) if data['sign'] == -1)

# print(f"Numero di archi positivi: {positive_edges}")
# print(f"Numero di archi negativi: {negative_edges}")

In [2]:
# Carica il grafo segnato (dalla repo del paper)
DATASET_PATH = "politisky24"  # esempio: "slashdot", "epinions", "twitterreferendum", "bitcoin", "highlandtribes"
signed_graph = SignedGraph(DATASET_PATH)

In [3]:
print("Numero di nodi nel grafo:", signed_graph.number_of_nodes)
print("Numero di archi nel grafo:", sum(len(neighbors[0]) + len(neighbors[1]) for neighbors in signed_graph.adjacency_list) // 2)

Numero di nodi nel grafo: 81951
Numero di archi nel grafo: 1566730


In [5]:
## Archi positivi e negativi con percentuale
positive_edges = sum(len(neighbors[0]) for neighbors in signed_graph.adjacency_list) // 2
negative_edges = sum(len(neighbors[1]) for neighbors in signed_graph.adjacency_list) // 2
total_edges = positive_edges + negative_edges
print(f"Numero di archi positivi: {positive_edges} ({positive_edges / total_edges * 100:.2f}%)")
print(f"Numero di archi negativi: {negative_edges} ({negative_edges / total_edges * 100:.2f}%)")

Numero di archi positivi: 1003636 (64.06%)
Numero di archi negativi: 563094 (35.94%)


In [15]:
# Definizione algoritmo LPA per risolvere problemi 2-PC
from dataclasses import dataclass
import heapq
from typing import Iterable, List, Optional, Set, Tuple
import numpy as np
from scipy.sparse import csr_matrix

# Parametri e stato

@dataclass
class LPAParams:
    allow_flips: bool = True           # se True, abilita flip su nodi già etichettati
    patience: int = 10                 # early stopping se f non migliora per 'patience' mosse accettate
    min_improv: float = 1e-8           # miglioramento minimo per azzerare la pazienza
    max_moves: Optional[int] = None    # tetto di sicurezza sulle mosse accettate (se None, impostato dinamicamente)
    pq_topK: Optional[int] = None      # limita la dimensione massima della PQ (None = illimitata)
    work_on_LCC: bool = True           # se True, lavora sulla LCC (unsigned) prima di partire
    log_every: int = 0                 # >0 per stampare progress ogni N mosse

@dataclass
class LPAState:
    A: csr_matrix          # matrice segnata in CSR --> formato per l'archiviazione di matrici sparse
    x: np.ndarray          # etichette correnti in {-1, 0, +1}
    q: np.ndarray          # q = A x
    g: float               # g = x^T A x
    s: int                 # s = x^T x (# nodi etichettati)
    version: int           # contatore di mosse accettate


# Utility di base

def sign_nonzero(z: float) -> int:
    """Ritorna +1, 0 o -1 come segno classico."""
    if z > 0: return  +1
    if z < 0: return  -1
    return 0

def safe_eval_f(g: float, s: int) -> float:
    """Valuta f=g/s con guardia per s=0. Restituisce -inf se s=0."""
    if s <= 0:
        return float("-inf")
    return g / s

def compute_initial_state(A: csr_matrix, x: np.ndarray) -> LPAState:
    """Costruisce q, g, s, version a partire da A e x."""
    q = A.dot(x)
    g = float(x.dot(q))
    s = int(x.dot(x))
    return LPAState(A=A, x=x, q=q, g=g, s=s, version=0)

# Accesso al grafo (da SignedGraph)

def get_neighbors_with_sign(sg, v: int) -> Iterable[Tuple[int, int]]:
    """
    Iteratore sui vicini (u, sigma) del nodo v.
    sigma = +1 per vicini positivi, -1 per vicini negativi.
    """
    pos_neighbors, neg_neighbors = sg.adjacency_list[v]
    for u in pos_neighbors:
        yield (u, +1)
    for u in neg_neighbors:
        yield (u, -1)

def has_labeled_neighbor(sg, x: np.ndarray, v: int) -> bool:
    """True se v ha almeno un vicino con x != 0."""
    for u, _sigma in get_neighbors_with_sign(sg, v):
        if x[u] != 0:
            return True
    return False

# Δf locali (stima ed esatte)

def estimate_add_gain(v: int, state: LPAState) -> Tuple[int, float]:
    """
    Stima del best add-gain per un nodo non etichettato:
    sceglie b = sign(q_v); se q_v = 0 restituisce un guadagno <= 0 (verrà scartato).
    Ritorna (b, estimated_gain).
    """
    s, g, qv = state.s, state.g, state.q[v]
    if s == 0:
        # primo inserimento: massimizza 2*b*qv
        b = +1 if qv >= 0 else -1
        return b, 2.0 * b * qv
    if qv > 0:
        # Δf = (2*qv*s - g) / (s*(s+1))
        return +1, (2.0 * qv * s - g) / (s * (s + 1))
    elif qv < 0:
        return -1, (-2.0 * qv * s - g) / (s * (s + 1))
    else:
        # se qv = 0, questa mossa raramente migliora f (stimiamo -g/(s*(s+1)) <= 0)
        return +1, (-g) / (s * (s + 1)) if s > 0 else 0.0

def exact_add_gain(v: int, b: int, state: LPAState) -> Tuple[float, float, int]:
    """
    Δf esatto per passare da x[v]=0 a x[v]=b in {+1,-1}.
    Ritorna (Δf, Δg, Δs).
    """
    qv = state.q[v]
    delta_g = 2.0 * b * qv
    delta_s = 1
    if state.s == 0:
        # f' = (g+Δg)/(s+1) con g=0,s=0 => f' = Δg/1 = Δg
        return delta_g, delta_g, delta_s
    delta_f = (delta_g * state.s - state.g * delta_s) / (state.s * (state.s + delta_s))
    return delta_f, delta_g, delta_s

def estimate_flip_gain(v: int, state: LPAState) -> float:
    """
    Stima del flip-gain per un nodo etichettato: a = x[v], b = -a.
    Δf = (-4*a*qv)/s (se s>0).
    """
    if state.s == 0:
        return float("-inf")
    a = state.x[v]
    qv = state.q[v]
    return (-4.0 * a * qv) / state.s

def exact_flip_gain(v: int, state: LPAState) -> Tuple[float, float, int]:
    """
    Δf esatto per flippare v: b = -x[v].
    Ritorna (Δf, Δg, Δs) con Δs = 0.
    """
    if state.s == 0:
        return float("-inf"), 0.0, 0
    a = state.x[v]
    qv = state.q[v]
    delta_g = -4.0 * a * qv
    delta_s = 0
    delta_f = delta_g / state.s
    return delta_f, delta_g, delta_s


# Applicazione mossa & aggiornamenti locali

def apply_move(sg, v: int, b: int, state: LPAState):
    """
    Applica la mossa (a -> b) su v, aggiornando x, g, s e q localmente.
    Nota: q[v] NON dipende da x[v] (A[v,v]=0), quindi non va aggiornato.
    """
    a = int(state.x[v])
    if a == b:
        return

    # aggiorna g e s con i delta già calcolati dal chiamante?
    # Qui per robustezza, li ricalcoliamo per consistenza:
    if a == 0:
        # add
        delta_g = 2.0 * b * state.q[v]
        delta_s = 1
    else:
        # flip
        delta_g = -4.0 * a * state.q[v]
        delta_s = 0

    state.x[v] = b
    state.g += delta_g
    state.s += delta_s

    # aggiorna q(u) per i vicini u di v: q[u] += A[u,v] * (b - a)
    delta = b - a
    if delta != 0:
        for u, sigma_uv in get_neighbors_with_sign(sg, v):
            state.q[u] += sigma_uv * delta

    # versione++ per invalidare le stime stale nella PQ
    state.version += 1


# Frontier & Priority Queue (CELF-like)

class CELFPQ:
    """
    Max-heap lazy: memorizza voci (key, v, move_type, move_label, version_snapshot).
    heapq è un min-heap, quindi usiamo key_neg = -key per il max.
    """
    __slots__ = ("heap", "size_cap")

    def __init__(self, size_cap: Optional[int] = None):
        self.heap: List[Tuple[float, int, str, int, int]] = []
        self.size_cap = size_cap

    def push(self, key: float, v: int, move_type: str, move_label: int, version: int):
        entry = (-key, v, move_type, move_label, version)
        heapq.heappush(self.heap, entry)
        if self.size_cap is not None and len(self.heap) > self.size_cap:
            # rimuove l'entry "peggiore" (chiave massima negativa => key più piccolo)
            heapq.heappop(self.heap)

    def pop_max(self) -> Optional[Tuple[float, int, str, int, int]]:
        if not self.heap:
            return None
        key_neg, v, move_type, move_label, version = heapq.heappop(self.heap)
        return (-key_neg, v, move_type, move_label, version)

    def empty(self) -> bool:
        return len(self.heap) == 0

    def __len__(self):
        return len(self.heap)

def build_frontier_and_pq(sg,
                          state: LPAState,
                          seed_mask: np.ndarray,
                          params: LPAParams) -> CELFPQ:
    """
    Costruisce la frontiera iniziale e popola la PQ con stime Δf.
    - Unlabeled con almeno un vicino etichettato -> candidate "add"
    - Se allow_flips: labeled con potenziale flip -> candidate "flip" (solo se gain stimato > 0)
    """
    n = state.x.shape[0]
    pq = CELFPQ(size_cap=params.pq_topK)

    for v in range(n):
        if seed_mask[v]:
            continue

        xv = state.x[v]
        if xv == 0:
            if has_labeled_neighbor(sg, state.x, v):
                b, est_gain = estimate_add_gain(v, state)
                pq.push(est_gain, v, "add", b, state.version)
        else:
            if params.allow_flips:
                est_gain = estimate_flip_gain(v, state)
                if est_gain > 0:
                    pq.push(est_gain, v, "flip", -xv, state.version)
    return pq

def refresh_candidates_after_move(sg,
                                  moved_v: int,
                                  state: LPAState,
                                  seed_mask: np.ndarray,
                                  params: LPAParams,
                                  pq: CELFPQ):
    """
    Dopo aver applicato una mossa su moved_v, aggiorna la PQ localmente (stile CELF):
    rivaluta SOLO i candidati nel vicinato (moved_v e i suoi vicini).
    """
    # 1) moved_v stesso
    v = moved_v
    if not seed_mask[v]:
        xv = state.x[v]
        if xv == 0:
            if has_labeled_neighbor(sg, state.x, v):
                b, est_gain = estimate_add_gain(v, state)
                pq.push(est_gain, v, "add", b, state.version)
        else:
            if params.allow_flips:
                est_gain = estimate_flip_gain(v, state)
                if est_gain > 0:
                    pq.push(est_gain, v, "flip", -xv, state.version)

    # 2) vicini di moved_v
    for u, _sigma in get_neighbors_with_sign(sg, moved_v):
        if seed_mask[u]:
            continue
        xu = state.x[u]
        if xu == 0:
            if has_labeled_neighbor(sg, state.x, u):
                b, est_gain = estimate_add_gain(u, state)
                pq.push(est_gain, u, "add", b, state.version)
        else:
            if params.allow_flips:
                est_gain = estimate_flip_gain(u, state)
                if est_gain > 0:
                    pq.push(est_gain, u, "flip", -xu, state.version)


# Preprocess & init labels

def largest_connected_component_nodes_unsigned(sg) -> np.ndarray:
    """
    Restituisce l'array (np.ndarray) degli indici dei nodi nella Largest Connected Component,
    considerando il grafo come **non segnato** (usa |A|).
    """
    # costruiamo l'adiacenza non segnata come |A|>0
    A = sg.get_adjacency_matrix()
    A_unsigned = A.copy()
    A_unsigned.data[:] = np.ones_like(A_unsigned.data)  # valori a 1
    # BFS/DFS su sparse: versione semplice via NetworkX se disponibile, altrimenti una DFS manuale
    # Per evitare dipendenze, facciamo una DFS manuale veloce.
    n = A_unsigned.shape[0]
    visited = np.zeros(n, dtype=bool)
    lcc_nodes: List[int] = []
    best_size = 0

    for start in range(n):
        if visited[start]:
            continue
        # stack DFS
        stack = [start]
        comp: List[int] = []
        visited[start] = True
        while stack:
            v = stack.pop()
            comp.append(v)
            row_start = A_unsigned.indptr[v]
            row_end = A_unsigned.indptr[v+1]
            neighbors = A_unsigned.indices[row_start:row_end]
            for u in neighbors:
                if not visited[u]:
                    visited[u] = True
                    stack.append(u)
        if len(comp) > best_size:
            best_size = len(comp)
            lcc_nodes = comp

    return np.array(sorted(lcc_nodes), dtype=int)

def extract_subgraph_by_nodes(sg, nodes: np.ndarray):
    """
    Ritorna una (sg_sub, mapping) dove sg_sub è un SignedGraph "compatibile"
    che condivide la stessa A in forma submatrice (via slicing CSR) e
    la stessa adjacency_list filtrata ai nodi 'nodes'.
    Per semplicità qui ritorniamo solo (A_sub, adj_sub) sufficienti per l'algoritmo.
    """
    # mappa vecchio->nuovo
    ordering = {int(v): i for i, v in enumerate(nodes)}
    # adjacency_list sub
    adj_sub = []
    for v in nodes:
        pos, neg = sg.adjacency_list[v]
        pos_f = [ordering[u] for u in pos if u in ordering]
        neg_f = [ordering[u] for u in neg if u in ordering]
        adj_sub.append([np.array(pos_f, dtype=int), np.array(neg_f, dtype=int)])

    # A_sub
    A = sg.get_adjacency_matrix().tocsr()
    A_sub = A[nodes, :][:, nodes].tocsr()
    # costruiamo un "wrapper" minimale con i campi che usiamo
    class SGView:
        def __init__(self, A_sub, adj_sub):
            self._A = A_sub
            self.adjacency_list = adj_sub
            self.number_of_nodes = A_sub.shape[0]
        def get_adjacency_matrix(self):
            return self._A
    return SGView(A_sub, adj_sub)

def init_labels(n: int, seeds_pos: Iterable[int], seeds_neg: Iterable[int]) -> np.ndarray:
    x = np.zeros(n, dtype=np.int8)
    for v in seeds_pos:
        x[v] = +1
    for v in seeds_neg:
        x[v] = -1
    return x

def build_seed_mask(n: int, seeds_pos: Iterable[int], seeds_neg: Iterable[int]) -> np.ndarray:
    mask = np.zeros(n, dtype=bool)
    for v in seeds_pos:
        mask[v] = True
    for v in seeds_neg:
        mask[v] = True
    return mask

import numpy as np
from typing import List, Tuple

def sample_seeds(signed_graph, frac: float = 0.01, seed: int = 42) -> Tuple[List[int], List[int]]:
    rng = np.random.default_rng(seed)

    n = signed_graph.number_of_nodes
    num_seeds = max(2, int(frac * n))   # almeno 2 seed
    half = num_seeds // 2

    all_nodes = np.arange(n)
    rng.shuffle(all_nodes)

    seeds_pos = all_nodes[:half].tolist()
    seeds_neg = all_nodes[half:num_seeds].tolist()

    print(f"Selected {len(seeds_pos)} positive and {len(seeds_neg)} negative seeds "
          f"out of {n} nodes (frac={frac:.2%})")

    return seeds_pos, seeds_neg


# MAIN: CELF-like LPA guidata dall'obiettivo 2-PC

def lpa_signed_2pc_celf(signed_graph, seeds_pos: Optional[Iterable[int]] = None, seeds_neg: Optional[Iterable[int]] = None,
                        params: Optional[LPAParams] = None, seed_frac: float = 0.01, seed_random: int = 42):
    """
    Algoritmo LPA semi-supervised CELF-like guidato dalla funzione obiettivo 2-PC (f = x^T A x / x^T x).
    - Usa SignedGraph per A (CSR) e adjacency_list per aggiornamenti locali.
    - I seed sono "clampati" e non cambiano mai.
    - Di default: no flip (allow_flips=False), espansione dai seed.
    """
    ## Test
    execution_time = ExecutionTime()

    if params is None:
        params = LPAParams()

    # 0) Preprocess: LCC (opzionale)
    sg = signed_graph
    # Se non sono stati forniti i seed, estraili casualmente
    if seeds_pos is None or seeds_neg is None:
        seeds_pos, seeds_neg = sample_seeds(signed_graph, frac=seed_frac, seed=seed_random)

    if params.work_on_LCC:
        nodes_lcc = largest_connected_component_nodes_unsigned(signed_graph)
        sg = extract_subgraph_by_nodes(signed_graph, nodes_lcc)
        # rimappi seeds sui nuovi indici
        index_map = {old: new for new, old in enumerate(nodes_lcc)}
        seeds_pos = [index_map[v] for v in seeds_pos if v in index_map]
        seeds_neg = [index_map[v] for v in seeds_neg if v in index_map]

    n = sg.number_of_nodes
    A = sg.get_adjacency_matrix().tocsr()


    # 1) Init labels & state
    x = init_labels(n, seeds_pos, seeds_neg)
    seed_mask = build_seed_mask(n, seeds_pos, seeds_neg)
    state = compute_initial_state(A, x)

    # 2) Init PQ (frontiera + stime)
    pq = build_frontier_and_pq(sg, state, seed_mask, params)

    # 3) Calcolo max_moves se non specificato, adattato alla dimensione del grafo
    m_est = int(A.nnz // 2)  # archi non orientati stimati (A simmetrica con 2 per edge)
    if params.max_moves is None:
        # tetto ragionevole: fino a 100*n o 20*m (il min)
        params.max_moves = min(100 * n, 20 * m_est)

    # 4) Loop greedy CELF-like
    f_best = safe_eval_f(state.g, state.s)
    patience_ctr = 0
    moves_done = 0

    while not pq.empty():
        top = pq.pop_max()
        if top is None:
            break
        key_est, v, move_type, b, ver_snapshot = top

        # Se l'entry è "stale" (è passata almeno una mossa da quando è stata stimata), ricalcola e reinserisci
        if ver_snapshot < state.version:
            if move_type == "add" and state.x[v] == 0 and not seed_mask[v]:
                # ricalcolo stima (lazy): basta reinserire con estimate attuale
                b_new, est_gain = estimate_add_gain(v, state)
                pq.push(est_gain, v, "add", b_new, state.version)
            elif move_type == "flip" and params.allow_flips and state.x[v] in (+1, -1) and not seed_mask[v]:
                est_gain = estimate_flip_gain(v, state)
                if est_gain > 0:
                    pq.push(est_gain, v, "flip", -state.x[v], state.version)
            continue  # passa al prossimo top

        # Entry nuova in PQ: calcolo Δf esatto
        if move_type == "add":
            if state.x[v] != 0 or seed_mask[v]:
                continue
            delta_f, delta_g, delta_s = exact_add_gain(v, b, state)
        elif move_type == "flip" and params.allow_flips:
            if state.x[v] not in (+1, -1) or seed_mask[v]:
                continue
            delta_f, delta_g, delta_s = exact_flip_gain(v, state)
        else:
            continue

        # Applica solo se migliora davvero
        if delta_f <= 0:
            continue

        # Applica la mossa (aggiorna x,g,s,q,version)
        apply_move(sg, v, b, state)
        moves_done += 1

        # Aggiorna i candidati nella zona impattata dalla mossa
        refresh_candidates_after_move(sg, v, state, seed_mask, params, pq)

        # Early stopping su f
        f_now = safe_eval_f(state.g, state.s)
        if f_now > f_best + params.min_improv:
            f_best = f_now
            patience_ctr = 0
        else:
            patience_ctr += 1

        # Log
        if params.log_every and (moves_done % params.log_every == 0):
            print(f"[moves={moves_done}] f={f_now:.6f}, s={state.s}, pq={len(pq)}")

        # Stop conditions
        if patience_ctr >= params.patience:
            print(f"No improvement in the last {params.patience} moves. Stopping (Patience).")
            break
        if moves_done >= params.max_moves:
            print(f"Reached max_moves={params.max_moves}. Stopping (Max Moves).")
            break    
    if pq.empty():
        print("Priority queue empty. No more improving moves. Stopping (PQ Empty).")

    # Fine: ritorna x, f_best e diagnostica
    diagnostics = {
        "n": n,
        "moves_done": moves_done,
        "final_f": safe_eval_f(state.g, state.s),
        "best_f": f_best,
        "labeled": int(state.s),
        "pq_left": len(pq),
    }
    execution_time.end_algorithm()
    print("Time LPA 2-PC:", execution_time.execution_time_seconds)
    return state.x, f_best, diagnostics

# 3) Parametri LPA (default ok per partire)
params = LPAParams(
    allow_flips=True,       # Permette di cambiare etichetta ai nodi già etichettati
    work_on_LCC=False,      # True consigliato per grafi molto sparsi
    patience=500,
    min_improv=1e-8,
    max_moves=None,         # calcolato in base a n e m --> Limitato comunque internamente
    pq_topK=None,           # None = illimitato
    log_every=100           # metti 1000 per logging periodico
)

### Esecuzione test

In [17]:
# 4) Esegui
FRAC_SEED_NODES = 0
x, best_f, diag = lpa_signed_2pc_celf(signed_graph, params=params, seed_frac=FRAC_SEED_NODES, seed_random=42)

# oppure, se passo anche i seed vector: seeds_pos, seeds_neg
# x, best_f, diag = lpa_signed_2pc_celf(signed_graph, seeds_pos, seeds_neg, params=params)

# 5) Risultati
print("Best f(x):", best_f)
print("Diagnostics:", diag)
print("Label counts: +1 =", int((x==1).sum()), " -1 =", int((x==-1).sum()), " 0 =", int((x==0).sum()))

# Info di base sul vettore x
print("type(x):", type(x), "dtype:", getattr(x, "dtype", None), "shape:", getattr(x, "shape", None))
print("unique in x:", np.unique(x), "...")
print("#nonzero:", np.count_nonzero(x))

# Costruisce gli insiemi S1 (+1) e S2 (-1)
S_1 = {i for i, val in enumerate(x) if val == 1}
S_2 = {i for i, val in enumerate(x) if val == -1}

print("\nS_1 (positivi):", len(S_1), "nodi")
print("Nodi S_1:", list(S_1)[:20], "..." if len(S_1) > 20 else "")

print("\nS_2 (negativi):", len(S_2), "nodi")
print("Nodi S_2:", list(S_2)[:20], "..." if len(S_2) > 20 else "")
## Confronto algoritmo Paper
### Ottenimento soluzione dagli algoritmi della repo
f_check = eigensign(signed_graph)
f_check_random = random_eigensign(signed_graph, beta='l1')

Selected 1 positive and 1 negative seeds out of 81951 nodes (frac=0.00%)
[moves=100] f=93.411765, s=102, pq=159979
[moves=200] f=179.089109, s=202, pq=289154
[moves=300] f=247.397351, s=302, pq=391429
[moves=400] f=297.164179, s=402, pq=461515
[moves=500] f=331.968127, s=502, pq=513087
[moves=600] f=356.647841, s=602, pq=576537
[moves=700] f=374.623932, s=702, pq=641170
[moves=800] f=387.286783, s=802, pq=685305
[moves=900] f=394.740576, s=902, pq=743815
[moves=1000] f=398.095808, s=1002, pq=770554
Priority queue empty. No more improving moves. Stopping (PQ Empty).
Time LPA 2-PC: 10.945
Best f(x): 398.7689463955638
Diagnostics: {'n': 81951, 'moves_done': 1080, 'final_f': np.float64(398.7689463955638), 'best_f': np.float64(398.7689463955638), 'labeled': 1082, 'pq_left': 0}
Label counts: +1 = 96  -1 = 986  0 = 80869
type(x): <class 'numpy.ndarray'> dtype: int8 shape: (81951,)
unique in x: [-1  0  1] ...
#nonzero: 1082

S_1 (positivi): 96 nodi
Nodi S_1: [70151, 75272, 28694, 26136, 16416,

In [14]:
# count di S_1 e S_2 dalla soluzione di eigensign e random_eigensign
S1_check = {i for i, val in enumerate(f_check[1]) if val == 1}
S2_check = {i for i, val in enumerate(f_check[1]) if val == -1}
print("\nEigensign - S_1 (positivi):", len(S1_check), "nodi")
print("Eigensign - S_2 (negativi):", len(S2_check), "nodi")

S_1_check_random = {i for i, val in enumerate(f_check_random[1]) if val == 1}
S_2_check_random = {i for i, val in enumerate(f_check_random[1]) if val == -1}
print("\nRandomEigensign - S_1 (positivi):", len(S_1_check_random), "nodi") 
print("RandomEigensign - S_2 (negativi):", len(S_2_check_random), "nodi")

# totali
print("Eigensign - Totale nodi:", len(S1_check) + len(S2_check), "nodi")
print("RandomEigensign - Totale nodi:", len(S_1_check_random) + len(S_2_check_random), "nodi")


Eigensign - S_1 (positivi): 92 nodi
Eigensign - S_2 (negativi): 1402 nodi

RandomEigensign - S_1 (positivi): 1024 nodi
RandomEigensign - S_2 (negativi): 3073 nodi
Eigensign - Totale nodi: 1494 nodi
RandomEigensign - Totale nodi: 4097 nodi
